# 07. Advanced Feature Engineering

**Objective:** Create 20+ predictive features from text, sentiment, entities, events, and topics

**Feature Categories:**
1. **Sentiment Features**: Momentum, volatility, directional changes
2. **Entity Features**: Frequency, diversity, co-mentions
3. **Event Features**: Impact scores, frequency, clustering
4. **Topic Features**: Diversity, concentration, shifts
5. **Temporal Features**: Lags, rolling windows, time-of-day
6. **Text Features**: Readability, urgency, linguistic patterns
7. **News Velocity**: Article frequency, burst detection

In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from collections import Counter, defaultdict
from tqdm.auto import tqdm
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Feature engineering
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from scipy import stats
from scipy.stats import skew, kurtosis

# Text analysis
import textstat
from textblob import TextBlob

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print(" All libraries loaded successfully")

 All libraries loaded successfully


## 1. Setup Paths and Configuration

In [4]:
# Setup paths
BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / '01_DATA'
FEATURES_DIR = DATA_DIR / 'features'
RESULTS_DIR = BASE_DIR / '03_RESULTS'
OUTPUTS_DIR = RESULTS_DIR / 'outputs'
VIZ_DIR = RESULTS_DIR / 'visualizations'
CONFIG_DIR = BASE_DIR / '06_CONFIG'

# Create directories
FEATURES_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
VIZ_DIR.mkdir(parents=True, exist_ok=True)

# Load configuration
with open(CONFIG_DIR / 'framework_config.json', 'r') as f:
    config = json.load(f)

print(f" Base directory: {BASE_DIR}")
print(f" Features directory: {FEATURES_DIR}")
print(f" Results directory: {RESULTS_DIR}")
print(f" Configuration loaded")

 Base directory: c:\Users\HARSHIT\Desktop\p\nlp analysis
 Features directory: c:\Users\HARSHIT\Desktop\p\nlp analysis\01_DATA\features
 Results directory: c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS
 Configuration loaded


## 2. Load Topic-Enriched Dataset

In [3]:
# Load dataset with topics from notebook 06
df = pd.read_csv(FEATURES_DIR / 'dataset_with_topics.csv')
df['date'] = pd.to_datetime(df['date'])

# Sort by date for time-series features
df = df.sort_values(['ticker', 'date']).reset_index(drop=True)

print(f" Loaded {len(df):,} articles")
print(f" Date range: {df['date'].min()} to {df['date'].max()}")
print(f" Companies: {df['ticker'].nunique()}")
print(f"\n Available features:")
print(f"  Columns: {df.shape[1]}")
df.head()

 Loaded 63 articles
 Date range: 2025-09-26 07:00:00 to 2026-01-18 12:19:48
 Companies: 19

 Available features:
  Columns: 60


,date,title,text,source,url,ticker,publisher,query,article_length,word_count,...,topic_0_prob,topic_1_prob,topic_2_prob,topic_3_prob,topic_4_prob,topic_5_prob,topic_6_prob,topic_7_prob,topic_8_prob,topic_9_prob
0,2026-01-14 08:02:00,This Unstoppable Stock Has 4 Catalysts to Fuel...,This Unstoppable Stock Has 4 Catalysts to Fuel...,Google News,https://news.google.com/rss/articles/CBMimAFBV...,AAPL,NaN,AAPL stock news,157,25,...,0.084952,0.178254,0.087264,0.083042,0.153865,0.081756,0.081756,0.081756,0.085599,0.081756
1,2026-01-18 12:15:12,Apple stock price slips to $255 ahead of a hol...,Apple stock price slips to $255 ahead of a hol...,Google News,https://news.google.com/rss/articles/CBMiuAFBV...,AAPL,NaN,AAPL stock news,121,18,...,0.027539,0.057783,0.028288,0.026919,0.725714,0.026502,0.026502,0.026502,0.027748,0.026502
2,2025-12-27 08:00:00,Is Weakness In Abbott Laboratories (NYSE:ABT) ...,Is Weakness In Abbott Laboratories (NYSE:ABT) ...,Google News,https://news.google.com/rss/articles/CBMiiwFBV...,ABT,NaN,ABT stock news,152,21,...,0.020583,0.800899,0.021143,0.020120,0.037280,0.019809,0.019809,0.019809,0.020740,0.019809
3,2025-12-31 08:00:00,Here is What to Know Beyond Why Abbott Laborat...,Here is What to Know Beyond Why Abbott Laborat...,Google News,https://news.google.com/rss/articles/CBMiiAFBV...,ABT,NaN,ABT stock news,102,15,...,0.020583,0.800899,0.021143,0.020120,0.037280,0.019809,0.019809,0.019809,0.020740,0.019809
4,2026-01-15 16:55:56,Dividend King Abbott Shows Why 52 Consecutive ...,Dividend King Abbott Shows Why 52 Consecutive ...,Google News,https://news.google.com/rss/articles/CBMi2gFBV...,ABT,NaN,ABT stock news,124,17,...,0.084952,0.178254,0.087264,0.083042,0.153865,0.081756,0.081756,0.081756,0.085599,0.081756


## 3. Sentiment Features

In [5]:
print(" Creating sentiment features...")

# 1. Sentiment momentum (rate of change)
df['sentiment_momentum'] = df.groupby('ticker')['finbert_sentiment'].diff()

# 2. Rolling sentiment averages
for window in [3, 7, 14]:
    df[f'sentiment_ma_{window}d'] = df.groupby('ticker')['finbert_sentiment'].transform(
        lambda x: x.rolling(window=window, min_periods=1).mean()
    )

# 3. Sentiment volatility
df['sentiment_volatility'] = df.groupby('ticker')['finbert_sentiment'].transform(
    lambda x: x.rolling(window=7, min_periods=1).std()
)

# 4. Sentiment range (max - min in window)
df['sentiment_range_7d'] = df.groupby('ticker')['finbert_sentiment'].transform(
    lambda x: x.rolling(window=7, min_periods=1).max() - x.rolling(window=7, min_periods=1).min()
)

# 5. Sentiment acceleration (change in momentum)
df['sentiment_acceleration'] = df.groupby('ticker')['sentiment_momentum'].diff()

# 6. Sentiment extremity (distance from neutral)
df['sentiment_extremity'] = df['finbert_sentiment'].abs()

# 7. Sentiment consensus (agreement between methods)
df['sentiment_consensus'] = (
    (df['finbert_sentiment'] * df['vader_compound']).apply(lambda x: 1 if x > 0 else 0)
)

print(" Sentiment features created:")
print("  - sentiment_momentum, sentiment_acceleration")
print("  - sentiment_ma_3d, sentiment_ma_7d, sentiment_ma_14d")
print("  - sentiment_volatility, sentiment_range_7d")
print("  - sentiment_extremity, sentiment_consensus")

 Creating sentiment features...
 Sentiment features created:
  - sentiment_momentum, sentiment_acceleration
  - sentiment_ma_3d, sentiment_ma_7d, sentiment_ma_14d
  - sentiment_volatility, sentiment_range_7d
  - sentiment_extremity, sentiment_consensus


## 4. Entity Features

In [6]:
print(" Creating entity features...")

# 1. Entity density (entities per word)
df['entity_density'] = df['num_entities'] / (df['text'].str.split().str.len() + 1)

# 2. Entity diversity (ratio of unique types)
df['entity_diversity'] = (
    (df['num_orgs'] > 0).astype(int) +
    (df['num_persons'] > 0).astype(int) +
    (df['num_locations'] > 0).astype(int)
) / 3

# 3. Organization prominence (ratio of org entities)
df['org_prominence'] = df['num_orgs'] / (df['num_entities'] + 1)

# 4. Rolling entity frequency
df['entity_freq_7d'] = df.groupby('ticker')['num_entities'].transform(
    lambda x: x.rolling(window=7, min_periods=1).sum()
)

# 5. Entity momentum
df['entity_momentum'] = df.groupby('ticker')['num_entities'].diff()

print(" Entity features created:")
print("  - entity_density, entity_diversity")
print("  - org_prominence, entity_freq_7d, entity_momentum")

 Creating entity features...
 Entity features created:
  - entity_density, entity_diversity
  - org_prominence, entity_freq_7d, entity_momentum


## 5. Event Features

In [7]:
print(" Creating event features...")

# 1. Event density
df['event_density'] = df['num_events'] / (df['text'].str.split().str.len() + 1)

# 2. High impact event indicator
df['has_high_impact_event'] = (df['event_impact_score'] > 0.7).astype(int)

# 3. Event frequency (rolling)
df['event_freq_7d'] = df.groupby('ticker')['num_events'].transform(
    lambda x: x.rolling(window=7, min_periods=1).sum()
)

# 4. Event momentum
df['event_momentum'] = df.groupby('ticker')['num_events'].diff()

# 5. Average event impact (rolling)
df['avg_event_impact_7d'] = df.groupby('ticker')['event_impact_score'].transform(
    lambda x: x.rolling(window=7, min_periods=1).mean()
)

# 6. Event-sentiment alignment
df['event_sentiment_alignment'] = df['event_impact_score'] * df['sentiment_extremity']

print(" Event features created:")
print("  - event_density, has_high_impact_event")
print("  - event_freq_7d, event_momentum")
print("  - avg_event_impact_7d, event_sentiment_alignment")

 Creating event features...
 Event features created:
  - event_density, has_high_impact_event
  - event_freq_7d, event_momentum
  - avg_event_impact_7d, event_sentiment_alignment


## 6. Topic Features

In [8]:
print(" Creating topic features...")

# Get topic probability columns
topic_cols = [col for col in df.columns if col.startswith('topic_') and col.endswith('_prob')]

if len(topic_cols) > 0:
    # 1. Topic concentration (entropy)
    def calculate_entropy(row):
        probs = [row[col] for col in topic_cols]
        probs = [p for p in probs if p > 0]
        if len(probs) == 0:
            return 0
        return -sum(p * np.log(p + 1e-10) for p in probs)
    
    df['topic_entropy'] = df.apply(calculate_entropy, axis=1)
    
    # 2. Topic dominance (max probability)
    df['topic_dominance'] = df[topic_cols].max(axis=1)
    
    # 3. Topic shift (change in dominant topic)
    df['topic_shift'] = (df.groupby('ticker')['dominant_topic'].diff() != 0).astype(int)
    
    # 4. Topic stability (rolling)
    df['topic_stability'] = 1 - df.groupby('ticker')['topic_shift'].transform(
        lambda x: x.rolling(window=7, min_periods=1).mean()
    )
    
    print(" Topic features created:")
    print("  - topic_entropy, topic_dominance")
    print("  - topic_shift, topic_stability")
else:
    print(" No topic probability columns found")
    df['topic_entropy'] = 0
    df['topic_dominance'] = 0
    df['topic_shift'] = 0
    df['topic_stability'] = 1

 Creating topic features...
 Topic features created:
  - topic_entropy, topic_dominance
  - topic_shift, topic_stability


## 7. Temporal Features

In [9]:
print(" Creating temporal features...")

# 1. Time-based features
df['hour'] = df['date'].dt.hour
df['day_of_week'] = df['date'].dt.dayofweek
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
df['is_market_hours'] = ((df['hour'] >= 9) & (df['hour'] <= 16)).astype(int)

# 2. Days since last article
df['days_since_last'] = df.groupby('ticker')['date'].diff().dt.total_seconds() / (24 * 3600)
df['days_since_last'] = df['days_since_last'].fillna(0)

# 3. Article recency score (exponential decay)
max_date = df['date'].max()
df['recency_score'] = np.exp(-(max_date - df['date']).dt.total_seconds() / (30 * 24 * 3600))

# 4. Lagged features
for lag in [1, 3, 7]:
    df[f'sentiment_lag_{lag}'] = df.groupby('ticker')['finbert_sentiment'].shift(lag)

print(" Temporal features created:")
print("  - hour, day_of_week, is_weekend, is_market_hours")
print("  - days_since_last, recency_score")
print("  - sentiment_lag_1, sentiment_lag_3, sentiment_lag_7")

 Creating temporal features...
 Temporal features created:
  - hour, day_of_week, is_weekend, is_market_hours
  - days_since_last, recency_score
  - sentiment_lag_1, sentiment_lag_3, sentiment_lag_7


## 8. News Velocity Features

In [10]:
print(" Creating news velocity features...")

# 1. Article count in rolling windows
df['article_count_1d'] = df.groupby('ticker')['date'].transform(
    lambda x: x.rolling('1D').count()
)
df['article_count_7d'] = df.groupby('ticker')['date'].transform(
    lambda x: x.rolling('7D').count()
)
df['article_count_30d'] = df.groupby('ticker')['date'].transform(
    lambda x: x.rolling('30D').count()
)

# 2. News acceleration (change in velocity)
df['news_acceleration'] = df.groupby('ticker')['article_count_7d'].diff()

# 3. News burst indicator (unusually high activity)
df['news_burst'] = (
    df['article_count_1d'] > df.groupby('ticker')['article_count_1d'].transform('mean') * 2
).astype(int)

# 4. Coverage consistency (std of article counts)
df['coverage_consistency'] = 1 / (1 + df.groupby('ticker')['article_count_7d'].transform('std').fillna(1))

print(" News velocity features created:")
print("  - article_count_1d, article_count_7d, article_count_30d")
print("  - news_acceleration, news_burst, coverage_consistency")

 Creating news velocity features...


ValueError: window must be an integer 0 or greater

## 9. Text Complexity Features

In [ ]:
print(" Creating text complexity features...")

def calculate_text_features(text):
    """Calculate readability and complexity metrics"""
    if pd.isna(text) or not isinstance(text, str) or len(text) < 10:
        return {
            'readability': 0,
            'word_count': 0,
            'avg_word_length': 0,
            'sentence_count': 0
        }
    
    try:
        # Flesch Reading Ease
        readability = textstat.flesch_reading_ease(text)
    except:
        readability = 50  # Default
    
    words = text.split()
    word_count = len(words)
    avg_word_length = np.mean([len(w) for w in words]) if words else 0
    
    # Count sentences
    sentence_count = text.count('.') + text.count('!') + text.count('?')
    
    return {
        'readability': readability,
        'word_count': word_count,
        'avg_word_length': avg_word_length,
        'sentence_count': max(sentence_count, 1)
    }

# Calculate text features
text_features = []
for text in tqdm(df['text'], desc="Text analysis"):
    text_features.append(calculate_text_features(text))

text_features_df = pd.DataFrame(text_features)

# Add to main dataframe
df['readability_score'] = text_features_df['readability']
df['word_count'] = text_features_df['word_count']
df['avg_word_length'] = text_features_df['avg_word_length']
df['sentence_count'] = text_features_df['sentence_count']
df['avg_sentence_length'] = df['word_count'] / df['sentence_count']

print(" Text complexity features created:")
print("  - readability_score, word_count, avg_word_length")
print("  - sentence_count, avg_sentence_length")

## 10. Composite Features

In [ ]:
print(" Creating composite features...")

# 1. News importance score (combination of multiple factors)
df['news_importance'] = (
    df['sentiment_extremity'] * 0.3 +
    df['event_impact_score'] * 0.3 +
    df['entity_density'] * 0.2 +
    df['topic_dominance'] * 0.2
)

# 2. Attention score (weighted combination)
df['attention_score'] = (
    df['article_count_7d'] * 0.4 +
    df['num_entities'] * 0.3 +
    df['num_events'] * 0.3
)

# 3. Signal strength (sentiment * impact * velocity)
df['signal_strength'] = (
    df['sentiment_extremity'] * 
    df['event_impact_score'] * 
    np.log1p(df['article_count_7d'])
)

# 4. Market relevance (composite score)
df['market_relevance'] = (
    df['is_market_hours'] * 0.3 +
    (1 - df['is_weekend']) * 0.3 +
    df['recency_score'] * 0.4
)

print(" Composite features created:")
print("  - news_importance, attention_score")
print("  - signal_strength, market_relevance")

## 11. Handle Missing Values and Infinities

In [ ]:
print(" Handling missing values and infinities...")

# Replace infinities with NaN
df = df.replace([np.inf, -np.inf], np.nan)

# Get all numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns

# Fill NaN values with median for each ticker
for col in numeric_cols:
    df[col] = df.groupby('ticker')[col].transform(
        lambda x: x.fillna(x.median())
    )
    # If still NaN, fill with global median
    df[col] = df[col].fillna(df[col].median())
    # If still NaN (all values were NaN), fill with 0
    df[col] = df[col].fillna(0)

# Check for remaining NaN
nan_counts = df[numeric_cols].isna().sum()
if nan_counts.sum() > 0:
    print(f" Columns with remaining NaN values:")
    print(nan_counts[nan_counts > 0])
else:
    print(" No missing values remaining")

# Check for infinities
inf_counts = np.isinf(df[numeric_cols]).sum()
if inf_counts.sum() > 0:
    print(f" Columns with infinity values:")
    print(inf_counts[inf_counts > 0])
else:
    print(" No infinity values remaining")

## 12. Feature Summary

In [11]:
# List all engineered features
engineered_features = [
    # Sentiment features (9)
    'sentiment_momentum', 'sentiment_ma_3d', 'sentiment_ma_7d', 'sentiment_ma_14d',
    'sentiment_volatility', 'sentiment_range_7d', 'sentiment_acceleration',
    'sentiment_extremity', 'sentiment_consensus',
    
    # Entity features (5)
    'entity_density', 'entity_diversity', 'org_prominence',
    'entity_freq_7d', 'entity_momentum',
    
    # Event features (6)
    'event_density', 'has_high_impact_event', 'event_freq_7d',
    'event_momentum', 'avg_event_impact_7d', 'event_sentiment_alignment',
    
    # Topic features (4)
    'topic_entropy', 'topic_dominance', 'topic_shift', 'topic_stability',
    
    # Temporal features (8)
    'hour', 'day_of_week', 'is_weekend', 'is_market_hours',
    'days_since_last', 'recency_score',
    'sentiment_lag_1', 'sentiment_lag_7',
    
    # News velocity features (6)
    'article_count_1d', 'article_count_7d', 'article_count_30d',
    'news_acceleration', 'news_burst', 'coverage_consistency',
    
    # Text complexity features (5)
    'readability_score', 'word_count', 'avg_word_length',
    'sentence_count', 'avg_sentence_length',
    
    # Composite features (4)
    'news_importance', 'attention_score', 'signal_strength', 'market_relevance'
]

# Filter to only features that exist
existing_features = [f for f in engineered_features if f in df.columns]

print(f" Feature Engineering Summary")
print("=" * 70)
print(f"Total engineered features: {len(existing_features)}")
print(f"\nFeature categories:")
print(f"  - Sentiment features: 9")
print(f"  - Entity features: 5")
print(f"  - Event features: 6")
print(f"  - Topic features: 4")
print(f"  - Temporal features: 8")
print(f"  - News velocity features: 6")
print(f"  - Text complexity features: 5")
print(f"  - Composite features: 4")
print(f"\n Target achieved: {len(existing_features)} features (target: 20+)")

# Save feature list
feature_metadata = {
    'total_features': len(existing_features),
    'feature_list': existing_features,
    'feature_categories': {
        'sentiment': 9,
        'entity': 5,
        'event': 6,
        'topic': 4,
        'temporal': 8,
        'news_velocity': 6,
        'text_complexity': 5,
        'composite': 4
    }
}

with open(OUTPUTS_DIR / 'feature_metadata.json', 'w') as f:
    json.dump(feature_metadata, f, indent=2)

print(f"\n Feature metadata saved to {OUTPUTS_DIR / 'feature_metadata.json'}")

 Feature Engineering Summary
Total engineered features: 33

Feature categories:
  - Sentiment features: 9
  - Entity features: 5
  - Event features: 6
  - Topic features: 4
  - Temporal features: 8
  - News velocity features: 6
  - Text complexity features: 5
  - Composite features: 4

 Target achieved: 33 features (target: 20+)

 Feature metadata saved to c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS\outputs\feature_metadata.json


## 13. Feature Statistics and Distributions

In [12]:
# Calculate statistics for engineered features
feature_stats = df[existing_features].describe().T
feature_stats['skewness'] = df[existing_features].apply(lambda x: skew(x))
feature_stats['kurtosis'] = df[existing_features].apply(lambda x: kurtosis(x))

print(" Feature Statistics (Top 20):")
print("=" * 100)
print(feature_stats.head(20))

# Save statistics
feature_stats.to_csv(OUTPUTS_DIR / 'feature_statistics.csv')
print(f"\n Full statistics saved to {OUTPUTS_DIR / 'feature_statistics.csv'}")

 Feature Statistics (Top 20):
                           count       mean       std        min       25%  \
sentiment_momentum          44.0   0.002936  0.524102  -1.379900 -0.361500   
sentiment_ma_3d             63.0   0.119636  0.282854  -0.571900 -0.011333   
sentiment_ma_7d             63.0   0.113599  0.281869  -0.571900 -0.033433   
sentiment_ma_14d            63.0   0.113599  0.281869  -0.571900 -0.033433   
sentiment_volatility        44.0   0.292986  0.185882   0.000000  0.192590   
sentiment_range_7d          63.0   0.397935  0.410162   0.000000  0.000000   
sentiment_acceleration      26.0   0.096581  1.058493  -1.692500 -0.606675   
sentiment_extremity         63.0   0.276544  0.246877   0.000000  0.000000   
sentiment_consensus         63.0   0.714286  0.455383   0.000000  0.000000   
entity_density              63.0   0.369333  0.126621   0.125000  0.266667   
entity_diversity            63.0   0.449735  0.190817   0.333333  0.333333   
org_prominence              63.0  

## 14. Feature Correlation Analysis

In [13]:
# Calculate correlation matrix for top features
top_features = existing_features[:20]  # Top 20 features
corr_matrix = df[top_features].corr()

# Visualize correlation heatmap
fig = px.imshow(
    corr_matrix,
    labels=dict(color="Correlation"),
    x=top_features,
    y=top_features,
    color_continuous_scale='RdBu_r',
    zmin=-1,
    zmax=1,
    title='Feature Correlation Matrix (Top 20 Features)'
)

fig.update_layout(
    height=800,
    xaxis_tickangle=-45
)

fig.write_html(VIZ_DIR / 'feature_correlation_matrix.html')
fig.show()

# Find highly correlated features (potential redundancy)
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > 0.8:
            high_corr_pairs.append((
                corr_matrix.columns[i],
                corr_matrix.columns[j],
                corr_matrix.iloc[i, j]
            ))

if high_corr_pairs:
    print(f"\n Highly correlated feature pairs (|r| > 0.8):")
    for feat1, feat2, corr in high_corr_pairs:
        print(f"  {feat1} <-> {feat2}: {corr:.3f}")
else:
    print("\n No highly correlated feature pairs found (good diversity)")

print(f"\n Correlation matrix saved to {VIZ_DIR / 'feature_correlation_matrix.html'}")


 Highly correlated feature pairs (|r| > 0.8):
  sentiment_momentum <-> sentiment_acceleration: 0.935
  sentiment_ma_3d <-> sentiment_ma_7d: 0.979
  sentiment_ma_3d <-> sentiment_ma_14d: 0.979
  sentiment_ma_7d <-> sentiment_ma_14d: 1.000
  sentiment_volatility <-> sentiment_range_7d: 0.896
  event_density <-> avg_event_impact_7d: 0.837
  event_density <-> event_sentiment_alignment: 1.000
  event_freq_7d <-> avg_event_impact_7d: 0.866
  avg_event_impact_7d <-> event_sentiment_alignment: 0.837

 Correlation matrix saved to c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS\visualizations\feature_correlation_matrix.html


## 15. Save Final Feature Matrix

In [14]:
# Save complete feature matrix
df.to_csv(FEATURES_DIR / 'final_feature_matrix.csv', index=False)

print(f" Final feature matrix saved to {FEATURES_DIR / 'final_feature_matrix.csv'}")
print(f" Dataset shape: {df.shape}")
print(f"   Rows: {df.shape[0]:,} articles")
print(f"   Columns: {df.shape[1]} features")
print(f"\n Engineered features: {len(existing_features)}")
print(f" Ready for machine learning modeling!")

df.head()

 Final feature matrix saved to c:\Users\HARSHIT\Desktop\p\nlp analysis\01_DATA\features\final_feature_matrix.csv
 Dataset shape: (63, 93)
   Rows: 63 articles
   Columns: 93 features

 Engineered features: 33
 Ready for machine learning modeling!


,date,title,text,source,url,ticker,publisher,query,article_length,word_count,...,topic_stability,hour,day_of_week,is_weekend,is_market_hours,days_since_last,recency_score,sentiment_lag_1,sentiment_lag_3,sentiment_lag_7
0,2026-01-14 08:02:00,This Unstoppable Stock Has 4 Catalysts to Fuel...,This Unstoppable Stock Has 4 Catalysts to Fuel...,Google News,https://news.google.com/rss/articles/CBMimAFBV...,AAPL,NaN,AAPL stock news,157,25,...,0.000000,8,2,0,0,0.000000,0.869966,NaN,NaN,NaN
1,2026-01-18 12:15:12,Apple stock price slips to $255 ahead of a hol...,Apple stock price slips to $255 ahead of a hol...,Google News,https://news.google.com/rss/articles/CBMiuAFBV...,AAPL,NaN,AAPL stock news,121,18,...,0.000000,12,6,1,1,4.175833,0.999894,-0.5719,NaN,NaN
2,2025-12-27 08:00:00,Is Weakness In Abbott Laboratories (NYSE:ABT) ...,Is Weakness In Abbott Laboratories (NYSE:ABT) ...,Google News,https://news.google.com/rss/articles/CBMiiwFBV...,ABT,NaN,ABT stock news,152,21,...,0.000000,8,5,1,0,0.000000,0.477425,NaN,NaN,NaN
3,2025-12-31 08:00:00,Here is What to Know Beyond Why Abbott Laborat...,Here is What to Know Beyond Why Abbott Laborat...,Google News,https://news.google.com/rss/articles/CBMiiAFBV...,ABT,NaN,ABT stock news,102,15,...,0.500000,8,2,0,0,4.000000,0.545521,-0.3818,NaN,NaN
4,2026-01-15 16:55:56,Dividend King Abbott Shows Why 52 Consecutive ...,Dividend King Abbott Shows Why 52 Consecutive ...,Google News,https://news.google.com/rss/articles/CBMi2gFBV...,ABT,NaN,ABT stock news,124,17,...,0.666667,16,3,0,1,15.372176,0.910640,0.0000,NaN,NaN


## 16. Summary Report

In [16]:
# Create comprehensive summary
summary = {
    'total_articles': len(df),
    'total_features': df.shape[1],
    'engineered_features': len(existing_features),
    'feature_categories': feature_metadata['feature_categories'],
    'date_range': {
        'start': str(df['date'].min()),
        'end': str(df['date'].max())
    },
    'companies_analyzed': int(df['ticker'].nunique()),
    'data_quality': {
        'missing_values': int(df[existing_features].isna().sum().sum()),
        'infinite_values': int(np.isinf(df[existing_features].select_dtypes(include=[np.number])).sum().sum()),
        'completeness': float(1 - df[existing_features].isna().sum().sum() / (len(df) * len(existing_features)))
    },
    'feature_statistics': {
        'mean_sentiment': float(df['finbert_sentiment'].mean()),
        'mean_impact': float(df['event_impact_score'].mean()) if 'event_impact_score' in df.columns else 0.0,
        'mean_importance': float(df['news_importance'].mean()) if 'news_importance' in df.columns else 0.0,
        'mean_signal_strength': float(df['signal_strength'].mean()) if 'signal_strength' in df.columns else 0.0
    }
}

# Save summary
with open(OUTPUTS_DIR / 'feature_engineering_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)

print(" Feature Engineering Summary")
print(f"Total articles: {summary['total_articles']:,}")
print(f"Total features: {summary['total_features']}")
print(f"Engineered features: {summary['engineered_features']}")
print(f"Companies: {summary['companies_analyzed']:,}")
print(f"\n Data quality:")
print(f"  Missing values: {summary['data_quality']['missing_values']:,}")
print(f"  Completeness: {summary['data_quality']['completeness']*100:.2f}%")
print(f"\n Key metrics:")
print(f"  Average sentiment: {summary['feature_statistics']['mean_sentiment']:.3f}")
if 'event_impact_score' in df.columns:
    print(f"  Average event impact: {summary['feature_statistics']['mean_impact']:.3f}")
if 'news_importance' in df.columns:
    print(f"  Average news importance: {summary['feature_statistics']['mean_importance']:.3f}")
if 'signal_strength' in df.columns:
    print(f"  Average signal strength: {summary['feature_statistics']['mean_signal_strength']:.3f}")

print(f"\n Summary saved to {OUTPUTS_DIR / 'feature_engineering_summary.json'}")
print("\n Feature Engineering complete!")
print("\n Output files:")
print(f"  - {FEATURES_DIR / 'final_feature_matrix.csv'}")
print(f"  - {OUTPUTS_DIR / 'feature_metadata.json'}")
print(f"  - {OUTPUTS_DIR / 'feature_statistics.csv'}")
print(f"  - {VIZ_DIR / 'feature_correlation_matrix.html'}")

 Feature Engineering Summary
Total articles: 63
Total features: 93
Engineered features: 33
Companies: 19

 Data quality:
  Missing values: 195
  Completeness: 90.62%

 Key metrics:
  Average sentiment: 0.115
  Average event impact: 0.007

 Summary saved to c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS\outputs\feature_engineering_summary.json

 Feature Engineering complete!

 Output files:
  - c:\Users\HARSHIT\Desktop\p\nlp analysis\01_DATA\features\final_feature_matrix.csv
  - c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS\outputs\feature_metadata.json
  - c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS\outputs\feature_statistics.csv
  - c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS\visualizations\feature_correlation_matrix.html
